# Data Preprocessing Test Notebook

This notebook tests preprocessing functions and shows clear output comparisons.

In [1]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'tests' else Path.cwd()
src_path = project_root / 'src'
dataset_dir = project_root / 'Dataset'
train_csv = dataset_dir / 'train.csv'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from preprocessing import (
    to_lower,
    remove_punctuation_and_digits,
    remove_stopwords,
    apply_stemming,
    preprocess_text,
    preprocess_dataframe,
)

print('Project root :', project_root)
print('Train exists :', train_csv.exists())

Project root : c:\Users\POOJA\OneDrive\Desktop\project
Train exists : True


## 1) Step-by-step preprocessing comparison on sample text

In [2]:
sample_text = 'This is A Sample Resume text, with 123 numbers and symbols!!! Running quickly.'

step_lower = to_lower(sample_text)
step_clean = remove_punctuation_and_digits(step_lower)
step_no_stopwords = remove_stopwords(step_clean)
step_stemmed = apply_stemming(step_no_stopwords)

steps_df = pd.DataFrame({
    'Stage': ['Original', 'Lowercase', 'Cleaned', 'Stopwords Removed', 'Stemmed'],
    'Text': [sample_text, step_lower, step_clean, step_no_stopwords, step_stemmed],
})
steps_df

,Stage,Text
0,Original,"This is A Sample Resume text, with 123 numbers..."
1,Lowercase,"this is a sample resume text, with 123 numbers..."
2,Cleaned,this is a sample resume text with numbers and ...
3,Stopwords Removed,sample resume text numbers symbols running qui...
4,Stemmed,sampl resum text number symbol run quickli


## 2) Validate preprocess_text output

In [3]:
final_output = preprocess_text(sample_text)
print('preprocess_text output:', final_output)
print('Matches final staged output:', final_output == step_stemmed)

preprocess_text output: sampl resum text number symbol run quickli
Matches final staged output: True


## 3) Real dataset comparison (first 5 rows)

In [4]:
raw_df = pd.read_csv(train_csv)
text_col = 'text' if 'text' in raw_df.columns else 'Text'

source_df = raw_df[[text_col]].copy().head(5)
source_df = source_df.rename(columns={text_col: 'text'})
processed_df = preprocess_dataframe(source_df, text_column='text')

comparison_df = pd.DataFrame({
    'Original Text': source_df['text'],
    'Processed Text': processed_df['text'],
})
comparison_df

,Original Text,Processed Text
0,ENGINEERING CONSULTANT Professional Summary To...,engin consult profession summari deliv valu pr...
1,CARPENTER Summary Carpenter Foreman Position w...,carpent summari carpent foreman posit effect u...
2,DIGITAL MARKETING SPECIALIST Summary Digital m...,digit market specialist summari digit market p...
3,SOUS CHEF Work Experience Sous Chef Jul 2010 C...,sou chef work experi sou chef jul compani ï¼​ ...
4,DONOR ADVOCATE Professional Summary Organized ...,donor advoc profession summari organ professio...


## 4) Clear numeric change comparison

In [5]:
metrics_df = pd.DataFrame({
    'Original Length': comparison_df['Original Text'].astype(str).str.len(),
    'Processed Length': comparison_df['Processed Text'].astype(str).str.len(),
    'Original Tokens': comparison_df['Original Text'].astype(str).str.split().str.len(),
    'Processed Tokens': comparison_df['Processed Text'].astype(str).str.split().str.len(),
})
metrics_df['Length Reduction'] = metrics_df['Original Length'] - metrics_df['Processed Length']
metrics_df['Token Reduction'] = metrics_df['Original Tokens'] - metrics_df['Processed Tokens']
metrics_df

,Original Length,Processed Length,Original Tokens,Processed Tokens,Length Reduction,Token Reduction
0,7959,5116,1133,777,2843,356
1,3165,2209,448,327,956,121
2,1987,1359,272,205,628,67
3,4806,3329,676,501,1477,175
4,5394,3473,773,496,1921,277
